## Previous floodfill implementation

In [ ]:
def get_neighborhood(z, y, x, shape):
    z, y, x = int(round(z)), int(round(y)), int(round(x))
    neighbors = []
    for i in range(-1, 2):
        for j in range(-1, 2):
            for k in range(-1, 2):
                if i + j + k == 0:
                    continue
                nz, ny, nx = z + k, y + j, x + i
                if 0 <= nz < shape[0] and 0 <= ny < shape[1] and 0 <= nx < shape[2]:
                    neighbors.append((nz, ny, nx))
    return neighbors

In [ ]:
from collections import deque
import numpy as np

def floodfill_opt(im, initial_mask, max_voxels=None, forbidden_mask=None):
    """
    Flood-fills a spine region across Z slices from sparse mask annotations,
    using intensity and 3D connectivity. Optionally excludes 'forbidden' voxels.

    Parameters:
    - im: 3D image stack (Z, Y, X)
    - initial_mask: 3D binary mask for current spine
    - max_voxels: optional cap on max number of voxels to include
    - forbidden_mask: 3D binary mask where fill is not allowed (e.g., dendrites)

    Returns:
    - final_mask: 3D binary mask after flood fill
    """
    final_mask = np.zeros_like(im, dtype=bool)
    final_mask[initial_mask] = True

    alpha = 0.8

    # Use a high-intensity voxel from the mask as seed
    zyx_coords = np.argwhere(initial_mask)
    seed_z, seed_y, seed_x = zyx_coords[np.argmax(im[initial_mask])]
    seed_intensity = float(im[seed_z, seed_y, seed_x])
    threshold = min(seed_intensity * alpha, np.percentile(im[initial_mask], 85))

    # Initialize queue
    queue = deque()
    queue.extend(get_neighborhood(seed_x, seed_y, seed_z, im.shape))

    while queue:
        x, y, z = queue.popleft()

        # Bounds check
        if not (0 <= z < im.shape[0] and 0 <= y < im.shape[1] and 0 <= x < im.shape[2]):
            continue

        # Already visited or forbidden (like dendrite)
        if final_mask[z, y, x]:
            continue
        if forbidden_mask is not None and forbidden_mask[z, y, x]:
            continue

        # Intensity check
        if im[z, y, x] > threshold:
            final_mask[z, y, x] = True
            neighbors = get_neighborhood(x, y, z, im.shape)
            for nx, ny, nz in neighbors:
                if forbidden_mask is not None and forbidden_mask[nz, ny, nx]:
                    continue
                if final_mask[nz, ny, nx]:
                    continue
                queue.append((nx, ny, nz))


            if max_voxels is not None and np.count_nonzero(final_mask) > max_voxels:
                print("Flood fill aborted: too many voxels")
                break

    return final_mask

## Utilities for improved centroid calculation

In [ ]:
from collections import deque
import numpy as np

# Medoid minimizes the sum of distances to the all foreground pixels
def get_medoid(mask):
    coords = np.argwhere(mask)
    dists = np.sum(np.linalg.norm(coords[:, None] - coords[None, :], axis=2), axis=1)
    medoid_index = np.argmin(dists)
    z_medoid, y_medoid, x_medoid = coords[medoid_index]
    return z_medoid, y_medoid, x_medoid

# Weighted median center (instead of weighted mean center, like the centroid) allways gives a
# center in the structure
def get_median(coords, weights):
    sort = np.argsort(coords)
    sorted = coords[sort]
    sorted_weights = weights[sort]
    cum_weights = np.cumsum(sorted_weights)
    total_weight = sorted_weights.sum()
    return sorted[cum_weights >= total_weight / 2][0]
def get_weighted_median_center(mask):
    z_coords, y_coords, x_coords = np.nonzero(mask)
    weights = mask[z_coords, y_coords, x_coords].astype(float)

    x_median = get_median(x_coords, weights)
    y_median = get_median(y_coords, weights)
    z_median = get_median(z_coords, weights)
    return z_median, y_median, x_median


## Improved floodfill

In [ ]:
from math import exp, floor
import matplotlib.pyplot as plt


def floodfill_opt_2(img, spine_mask, max_voxels=None, forbidden_mask=None, alpha=0.85):
    """Flood-fills spine mask labeled with 'label_id' across Z slices from sparse mask annotations,
    using intensity and 3D connectivity. Optionally excludes 'forbidden' voxels.

    Args:
        img (array): 3D image stack (Z, Y, X).
        spine_mask (array): 3D binary mask for current spine.
        max_voxels (int): optional cap on max number of voxels to include.
        forbidden_mask (array): 3D binary mask where fill is not allowed (e.g. dendrite).
        alpha (float): parameter that determines the cut on intensity for a voxel to be chosen as seed.
                       Default to 0.85. Should belong to [0, 1].

    Returns:
        array: 3D binary mask after flood-fill.
    """
    # Sanity check
    alpha = max(min(alpha, 1.0), 0.0)
    if forbidden_mask is None:
        forbidden_mask = np.zeros_like(img, dtype=bool)

    visited_mask = np.zeros_like(img, dtype=bool)
    final_mask = np.zeros_like(img, dtype=bool)
    final_mask[spine_mask] = True

    # Spines sometimes have complex shapes, so the centroid may fall in background. Use a center
    # calculated from the weighted medians instead.
    centroid = list(get_weighted_median_center(spine_mask)) # = [seed_z, seed_y, seed_x]

    # Get centroid's brightness to seed the floodfilling. Because centroid is float, we round to integer
    # and taking the average brightness between points resulting from ceil and floor.
    seeds = []
    intensities = []
    rounding_combinations = [(0, 0), (0, 1), (1, 0), (1, 1)]
    for round_y, round_x in rounding_combinations:
        rounded_centroid = centroid.copy()
        rounded_centroid[0] = floor(rounded_centroid[0])
        rounded_centroid[1] = floor(rounded_centroid[1]) + round_y
        rounded_centroid[2] = floor(rounded_centroid[2]) + round_x
        z, y, x = rounded_centroid
        intensity = img[z, y, x]
        # Spines sometimes have complex shapes, so the medoid's rounded neightbors may fall in background
        if abs(intensity) != 0:
            seeds.append(rounded_centroid)
            z, y, x = rounded_centroid
            intensities.append([intensity])

    centroid_brightness = np.mean(intensities) - np.std(intensities)
    
    # Initialize queue
    queue = deque()
    for seed in seeds:
        queue.extend(get_neighborhood(seed[0], seed[1], seed[2], img.shape))

    # Never allow a too low threshold brightness
    zyx_coords = np.argwhere(spine_mask)
    max_z, max_y, max_x = zyx_coords[np.argmax(img[spine_mask])]
    max_brightness = float(img[max_z, max_y, max_x])
    centroid_percentage = centroid_brightness / max_brightness
    # Use y-axis-flipped sigmoid to only assign higher minimum intensities to centroids with low
    # intensity ratio wrt maximum
    sig_percentage = 1 / (1 + exp(centroid_percentage))
    threshold = max(centroid_brightness * alpha, max_brightness * sig_percentage)

    # Loop over neighbours
    while queue:
        z, y, x = queue.popleft()

        # Already visited or forbidden voxels are discarded, as well as too far voxels
        if visited_mask[z, y, x]:
            continue
        if forbidden_mask[z, y, x]:
            continue
        # Add distance constraint

        visited_mask[z, y, x] = True

        # If intense enough, the neighbour is added to the mask and its neighbours to the queue
        if img[z, y, x] > threshold:
            plt.scatter(x, y, c='g', s=10)
            final_mask[z, y, x] = True
            neighbors = get_neighborhood(z, y, x, img.shape)
            for nz, ny, nx in neighbors:
                queue.append((nz, ny, nx))

            # if max_voxels is not None and np.count_nonzero(final_mask) > max_voxels:
            #     print(
            #         f"Floodfill aborted: number of voxels added ({np.count_nonzero(final_mask)}) exceeded max_voxels ({max_voxels})"
            #     )
            #     break

    return final_mask


In [ ]:
import numpy as np
import imageio as io
import matplotlib.pyplot as plt
from skimage.measure import label

img_folder = "test_images"
for animal in {"turtles"}:  # , "mice"}:
    stack = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_orig.tif"))
    spines = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_spines.tif"))
    dendrite = np.asarray(io.mimread(
        f"{img_folder}/out/{animal}_dendrite.tif"))

    # Label spines
    L = label(spines)

    # Get all spine labels (exclude background = 0)
    labels = np.unique(L)
    labels = labels[labels != 0]

    # Final mask to accumulate filled results
    final_mask = np.zeros_like(L, dtype=bool)

    # Optional: store diffs to analyze floodfill additions
    diff_map = np.zeros_like(L, dtype=np.int8)

    # Get dendrite mask and extend it over the whole Z axis for avoiding floodfilling spines into dendrite
    dendrite_mask = dendrite == 255
    dendrite_mask_combined = np.any(dendrite_mask, axis=0)
    dendrite_mask_broad = np.broadcast_to(dendrite_mask_combined, dendrite_mask.shape)
    forbidden_mask = dendrite_mask_broad

    # Loop through all spines
    for label_id in labels:
        spine_mask = L == label_id
        print(f"processing label {label_id}")

        # if label_id == 14 or label_id == 17 or label_id == 19 label_id == 21 or label_id == 27:
        plt.figure()
        plt.imshow(spine_mask.max(0), cmap='grey')
        ys, xs = np.where(spine_mask.max(0))
        plt.xlim(xs.min()-100, xs.max()+100)
        plt.ylim(ys.max()+100, ys.min()-100)
        # else:
        #   continue

        filled = floodfill_opt_2(img=stack, spine_mask=spine_mask, forbidden_mask=forbidden_mask)
        plt.show()
        # Accumulate into final mask
        final_mask |= filled

        # Optional: difference map
        diff = filled.astype(int) - spine_mask.astype(int)
        diff_map += diff.astype(np.int8)  # accumulate all diffs

    io.mimwrite(f"{img_folder}/out/{animal}_spines_flooded.tif",
                final_mask.astype(np.uint8) * 255)
    
    # Save or visualize final mask
    plt.figure()
    plt.imshow(final_mask.max(0), cmap='gray')
    plt.title(f"Final {animal} accumulated mask (max projection)")
    plt.axis('off')
    plt.show()

    # Show total diff (accumulated)
    plt.figure()
    plt.imshow(diff_map.max(0), cmap='bwr', vmin=-1, vmax=1)
    plt.title(f"Total {animal} difference map: red = added, blue = lost")
    plt.axis('off')
    plt.show()


## Show overlay for comparing new and old labels

In [ ]:
L_flood = label(final_mask)
    
for label_id in labels:
    old_mask = (L == label_id)

    # Find the new labels that overlap with the old ones (ignorinig bg)
    overlapping_new_labels = np.unique(L_flood[old_mask])
    print(f"Label {label_id} now is {overlapping_new_labels}")
    overlapping_new_labels = overlapping_new_labels[overlapping_new_labels != 0]

    new_mask = np.isin(L_flood, overlapping_new_labels)

    old_mask_proj = old_mask.max(axis=0)
    new_mask_proj = new_mask.max(axis=0)
    diff = new_mask_proj & ~old_mask_proj

    # Plot overlay for comparison: red for old and blue for new
    overlay = np.zeros(old_mask_proj.shape + (3,), dtype=np.uint8)
    overlay[..., 0] = old_mask_proj * 255
    overlay[..., 2] = diff * 255
    plt.imshow(overlay)
    ys, xs = np.where(old_mask_proj)
    plt.xlim(xs.min()-100, xs.max()+100)
    plt.ylim(ys.max()+100, ys.min()-100)
    plt.title(
        f"Comparison for label {label_id} (red = original, blue = new with floodfill)")
    plt.axis('off')
    plt.show()

# Plot general overlay
# * red = original
# * blue = added masks with floodfill
# * pink/purple-ish = re-labeled masks with floodfill
L_proj = (L > 0).max(axis=0)
diff_proj = diff_map.max(axis=0)
overlay = np.zeros(L_proj.shape + (3,), dtype=np.uint8)
overlay[..., 0] = L_proj * 255
overlay[..., 2] = diff_proj * 255

plt.figure()
plt.imshow(overlay)
plt.title("General overview")
plt.axis('off')
plt.show()

## Create composite

In [ ]:
import cv2
import numpy as np
import tifffile

orig_img = stack
spines_mask = (final_mask > 0).astype(np.uint8) * 255
dendrite_mask = (dendrite_mask > 0).astype(np.uint8) * 255

flooded_overlay = []

for z in range(orig_img.shape[0]):
    s_mask = spines_mask[z]
    d_mask = dendrite_mask[z]
    orig_img_rgb = cv2.cvtColor(orig_img[z].astype(np.uint8), cv2.COLOR_GRAY2BGR)

    color_mask = np.zeros_like(orig_img_rgb)
    color_mask[:, :, 0] = d_mask
    color_mask[:, :, 2] = d_mask
    color_mask[:, :, 1] = s_mask

    overlay = cv2.addWeighted(orig_img_rgb, 1.0, color_mask, 0.5, 0)
    flooded_overlay.append(overlay)

flooded_overlay = np.stack(flooded_overlay, axis=0)

tifffile.imwrite(f"{img_folder}/out/{animal}_spines_flooded_overlay.tif",
    flooded_overlay,
    photometric="rgb"
)
